In [1]:
using LinearAlgebra

function reck_decomposition(U_input::AbstractMatrix{<:Number})
    U = Matrix{ComplexF64}(U_input)
    N = size(U, 1)
    transformations = []

   
    for col in 1:(N-1)
        for row in N:-1:(col+1)
            target = U[row, col]
            pivot  = U[row-1, col]
            
          
            if abs(target) < 1e-12
                continue
            end
            
            r = sqrt(abs2(pivot) + abs2(target))
            cos_theta = abs(pivot) / r
            sin_theta = abs(target) / r
            
            phi = angle(target) - angle(pivot)
            theta = atan(sin_theta, cos_theta)
            
            T_inv = [
                 cos_theta                   sin_theta * exp(-im * phi);
                -sin_theta * exp(im * phi)   cos_theta
            ]
          
            U[row-1:row, :] .= T_inv * U[row-1:row, :]
            
            push!(transformations, (row-1, row, theta, phi))
        end
    end
    
    phases = angle.(diag(U))
    return transformations, phases
end

reck_decomposition (generic function with 1 method)

In [2]:
function Fourier(m)
    F=zeros(Complex,m,m)
    for i=0:m -1
        for j=0:m-1
            F[i+1,j+1]=exp(2*π*i*j*im/m)
        end
    end
    return F/sqrt(m)
end

function reconstruct_unitary(transformations, phases)
    N = length(phases)
    
    U_rec = zeros(ComplexF64, N, N)
    for i in 1:N
        U_rec[i, i] = exp(im * phases[i])
    end
    
    for (m1, m2, theta, phi) in reverse(transformations)
        cos_t = cos(theta)
        sin_t = sin(theta)
       
        T = [
             cos_t                   -sin_t * exp(-im * phi);
             sin_t * exp(im * phi)    cos_t
        ]
        
        U_rec[[m1, m2], :] .= T * U_rec[[m1, m2], :]
    end
    
    return U_rec
end


function T(θ,ϕ,a,b,n)
    U=diagm(ones(Complex,n))
    T = [
             cos(θ)                   -sin(θ) * exp(-im * ϕ);
             sin(θ) * exp(im * ϕ)    cos(θ)
        ]
    U[a,a],U[b,b]=T[1,1],T[2,2]
    U[a,b],U[b,a]=T[1,2],T[2,1]
    return U 
end

P(ϕ)=diagm(exp.(im*ϕ))

P (generic function with 1 method)

In [24]:
Perm=[0 0 -1; 1 0 0; 0 1 0] # Perm=[0 0 -1; 1 0 0; 0 1 0]
D,U=eigen(Perm)
U*diagm(D)*U'

3×3 Matrix{ComplexF64}:
          0.0+0.0im   2.77556e-16+0.0im          -1.0+0.0im
          1.0+0.0im  -3.88578e-16+0.0im   2.77556e-16+0.0im
 -5.55112e-17+0.0im           1.0+0.0im  -1.11022e-16+0.0im

In [25]:

bs_sequence, final_phases = reck_decomposition(U)

println("--- Optical Component Sequence ---")
for (step, (m1, m2, theta, phi)) in enumerate(bs_sequence)
    println("Step $step: Beam Splitter/Phase Shifter on modes ($m1, $m2)")
    println("   Theta (Transmissivity): $(round(cos(theta)^2, digits=4))")
    println("   Phi (Phase shift):      $(round(mod(phi,2π), digits=4)) rad")
end

println("\n--- Final Phase Shifts (Diagonal D) ---")
for (mode, phase) in enumerate(final_phases)
    println("Mode $mode: $(round(phase, digits=4)) rad")
end

--- Optical Component Sequence ---
Step 1: Beam Splitter/Phase Shifter on modes (2, 3)
   Theta (Transmissivity): 0.5
   Phi (Phase shift):      3.1416 rad
Step 2: Beam Splitter/Phase Shifter on modes (1, 2)
   Theta (Transmissivity): 0.3333
   Phi (Phase shift):      3.1416 rad
Step 3: Beam Splitter/Phase Shifter on modes (2, 3)
   Theta (Transmissivity): 0.5
   Phi (Phase shift):      1.5708 rad

--- Final Phase Shifts (Diagonal D) ---
Mode 1: 0.0 rad
Mode 2: 1.0472 rad
Mode 3: -2.618 rad


In [30]:
norm(P([0,0,π])*T(π/4,0,2,3,3)*P([0,0,π])*P([0,π,0])*T(acos(1/sqrt(3)),0,1,2,3)*P([0,π,0])*P([0,0,π/2])*T(π/4,0,2,3,3)*P([0,0,-π/2])*P([0,π/3,-5*π/6])-U,1)
#norm(P([0,0,2*π/3])*T(π/4,0,2,3,3)*P([0,0,-2*π/3])*P([0,2*π/3,0])*T(acos(1/sqrt(3)),0,1,2,3)*P([0,-2*π/3,0])*P([0,0,7*π/6])*T(π/4,0,2,3,3)*P([0,0,-7*π/6])*P([-π/3,0,5*π/6])-U,1)

1.7227022316728667e-15

In [27]:
F = P([-5*π/6, -2*π/3, 0]) * T(π/4, π/2, 1, 2, 3) * T(asin(1/sqrt(3)), π/2, 1, 3, 3) * T(π/4, π/2, 2, 3, 3) * P([-π/2, -π/2, 0])
F' * P([2*π/3, 4*π/3, 0]) * F

3×3 Matrix{Complex}:
         0.0+2.22045e-16im  …          1.0-2.7242e-16im
         1.0-3.33067e-16im     1.66533e-16+2.37494e-16im
 2.22045e-16+2.77556e-17im     5.55112e-17+1.54494e-17im

In [28]:
D

3-element Vector{ComplexF64}:
               -1.0 + 0.0im
 0.4999999999999998 - 0.8660254037844383im
 0.4999999999999998 + 0.8660254037844383im

In [29]:
sqrt(3)/2

0.8660254037844386